In [3]:
import sys
import os
import numpy as np
import pandas as pd
import scipy.constants as spc
from scipy.optimize import minimize
from scipy.optimize import curve_fit
from scipy.optimize import fsolve
from scipy.optimize import brentq
from astropy.wcs import WCS
from astropy.io import fits 
from astropy.io import ascii
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy import constants as const
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from numpy import linalg as LA
from IPython.display import HTML
from sklearn.model_selection import ParameterGrid
from scipy.interpolate import interp1d
from gofish import imagecube
import time
import matplotlib.animation as animation
from mpl_toolkits.axes_grid1 import make_axes_locatable
import PSSpy as pss
import imdata
sys.path.append(os.getcwd()) # change the name of directory.
sys.path.append(os.path.abspath("Imfits"))
from imfits import Imfits


In [4]:
ra_start = '03:32:13.273'
ra_end = '03:32:22.627'
dec_deg = 30 + 49/60 + 47.7050/3600  # +31.21.38.180 => 31.3606 度
distance_pc = 300
radius_ref_au = 1430 #Disk edge (au)
n_pixels = 256
marker_size = 2
arcsec_range, AU_per_pixel = pss.calc_ra_arcsec(ra_start, ra_end, dec_deg, distance_pc, n_pixels)

print(f"RA arcsec range (rounded): {arcsec_range} arcsec")
print(f"AU per pixel: {AU_per_pixel:.2f} AU/pixel")

RA arcsec range (rounded): 120 arcsec
AU per pixel: 140.63 AU/pixel


In [5]:
# 定義參數
Local_Standard_Velocity = 7 #km/s
pa_deg = 20 + 90
pa_rad = pa_deg * np.pi / 180
# ----------- input -----------
f_mom0 = "Per-emb-2-HC3N_10-9_TdV.fits"
f_mom1 = "Per-emb-2-HC3N_10-9_fit_Vc.fits"

im_mom0 = Imfits(f_mom0)
im_mom1 = Imfits(f_mom1)

# data
data_mom0 = im_mom0.data
data_mom1 = im_mom1.data

x = np.arange(256) - 128
z = np.arange(256) - 128
xx, zz = np.meshgrid(x, z)
r, theta = pss.spherical_coords(xx, zz)

N_elements = 11
pars = np.linspace(18,50,N_elements+1)  # Define distance ranges
xmeans = np.zeros(N_elements)
zmeans = np.zeros(N_elements)
xzstd = np.zeros(N_elements)
x_means = np.zeros(N_elements)
z_means = np.zeros(N_elements)
v_means = np.zeros(N_elements)
pcstds = np.zeros((2,N_elements))  # intialize "empty" array
x_array_list = []
z_array_list = []
v_array_list = []
weights_list = []

for i in range(N_elements):
    dinds = (r>pars[i]) & (r<=pars[i+1])  # Identify points in given distance range
    xmeans[i] = np.average(xx[dinds],weights=data_mom0[dinds])  # Add means
    zmeans[i] = np.average(zz[dinds],weights=data_mom0[dinds])  # Add means    
    xzstd[i] = np.sqrt(np.average((xx[dinds] - xmeans[i]) ** 2 + (zz[dinds] - zmeans[i]) ** 2, weights=data_mom0[dinds]))
    
r_means, theta_means = pss.spherical_coords(xmeans,zmeans)

theta_r = interp1d(r_means, theta_means, fill_value=(theta_means[0], theta_means[-1]), bounds_error=False) #
std_r = interp1d(r_means, xzstd, fill_value=(xzstd[0], xzstd[-1]), bounds_error=False)

mom1_vel = np.ma.masked_invalid(data_mom1)

for i in range(N_elements):
    r_ref = (pars[i]+ pars[i+1]) / 2
    theta_ref = theta_r(r_ref)
    std_ref = std_r(r_ref) / r_ref
    
    delta_theta = np.pi-np.abs(np.pi-np.abs(theta-theta_ref))
    weights = data_mom0 * pss.gaussian(delta_theta, 0, std_ref)
    
    dinds = (r>pars[i]) & (r<=pars[i+1])  # Identify points in given distance range
    dinds_v = (r>pars[i]) & (r<=pars[i+1]) & np.isfinite(data_mom1)
    
    # 存儲每次迴圈的值
    x_array_list.append(xx[dinds])
    z_array_list.append(zz[dinds])
    v_array_list.append(mom1_vel[dinds])
    weights_list.append(weights[dinds])
    
    x_means[i] = np.average(xx[dinds],weights=weights[dinds])  # Add means  
    z_means[i] = np.average(zz[dinds],weights=weights[dinds])  # Add means
    v_means[i] = np.average(mom1_vel[dinds_v], weights=weights[dinds_v]) #
v_means_LS_m = (v_means - Local_Standard_Velocity) * 1e3 # 以影像中心 (128, 128) 為基準
v_means_LS_km = v_means - Local_Standard_Velocity # 以影像中心 (128, 128) 為基準

# for i in np.linspace(0, 40, 10):
#     pa_deg = i + 90
#     pa_rad = np.deg2rad(pa_deg)
#     # **將Fitting 參數轉換成 AU**
#     x_means_m = (x_means * np.cos(pa_rad) + z_means * np.sin(pa_rad)) * AU_per_pixel * spc.astronomical_unit # 以影像中心 (128, 128) 為基準
#     z_means_m = (-x_means * np.sin(pa_rad) + z_means * np.cos(pa_rad)) * AU_per_pixel * spc.astronomical_unit # 以影像中心 (128, 128) 為基準


#     R = v_means_LS_m / x_means_m
#     X_r = x_means_m * R ** (3/4)
#     Z_r = z_means_m * R ** (3/4)
#     R = R ** (1/4)

#     plt.plot(R[:-3], X_r[:-3])
#     plt.plot(R[:-3], Z_r[:-3])
#     plt.show()
# plt.plot(R, A * np.cos(k_array( R + R0)))
# print(X_r)
# print(Z_r)
# print(R)
x_means_m = (x_means * np.cos(pa_rad) + z_means * np.sin(pa_rad)) * AU_per_pixel * spc.astronomical_unit # 以影像中心 (128, 128) 為基準
z_means_m = (-x_means * np.sin(pa_rad) + z_means * np.cos(pa_rad)) * AU_per_pixel * spc.astronomical_unit # 以影像中心 (128, 128) 為基準

R = v_means_LS_m / x_means_m
X_r = x_means_m * R ** (3/4)
Z_r = z_means_m * R ** (3/4)
R = R ** (1/4)

CAUTION	channelmap: No keyword PCi_j or CDi_j are found. No rotation is assumed.
CAUTION	channelmap: No keyword PCi_j or CDi_j are found. No rotation is assumed.


In [ ]:
nk = 100
nR = 80
n_min = 10
k_array = np.linspace(0.01, 2, nk) * np.pi / (np.max(R) - np.min(R))
R_0f = np.zeros_like(k_array)
R_0g = np.zeros_like(k_array)
R_0_array = np.linspace(0, np.pi / k_array[0], nR)
f_kR_re = np.zeros_like(R_0_array)
g_kR_re = np.zeros_like(R_0_array)
Rf_root = np.zeros([nk, 4])
Rg_root = np.zeros([nk, 4])
gf_root = np.ones([nk, 4])
gg_root = np.ones([nk, 4])
params = [X_r[:-3], Z_r[:-3], R[:-3]]
Rf_temp = np.zeros(4)
Rg_temp = np.zeros(4)
chi_square = np.zeros((nk, nR))
clac_Inclination = np.zeros((nk, nR))
clac_phi_par = np.zeros((nk, nR))
clac_theta_par = np.zeros((nk, nR))
clac_omega_ref = np.zeros((nk, nR))
clac_t = np.zeros((nk, nR))
chi_square_min = np.zeros((4, n_min, 2))
inds_f = [0, 1]
inds_g = [0, 1]

for ik in range(nk)[::1]:
    R_0_array = np.linspace(0, np.pi / k_array[ik], nR)
    for i in range(nR):
        chi_square[ik, i] = pss.chi_square(R_0_array[i], k_array[ik], params)
        A, B, C, clac_Inclination[ik, i], clac_phi_par[ik, i], clac_theta_par[ik, i], clac_omega_ref[ik, i], clac_t[ik, i] = pss.calc_params(R_0_array[i], k_array[ik], params, radius_ref_au)


for ik in range(nk)[::1]:
    R_0_array = np.linspace(0, np.pi / k_array[ik], nR)
    for i in range(nR):
        f_kR_re[i] = pss.f_kR(R_0_array[i], k_array[ik], params)
        g_kR_re[i] = pss.g_kR(R_0_array[i], k_array[ik], params)
        # chi_square[ik, i] = pss.chi_square(R_0_array[i], k_array[ik], params)
    plt.plot(R_0_array * k_array[ik], f_kR_re / np.max(f_kR_re))
    plt.plot(R_0_array * k_array[ik], g_kR_re / np.max(g_kR_re), '--')
    f_ind = np.where(f_kR_re[:-1] * f_kR_re[1:]<0)[0]
    g_ind = np.where(g_kR_re[:-1] * g_kR_re[1:]<0)[0]
    # print(g_ind)

    # n_root = len(f_ind)
    for i in range(len(f_ind)):
        R_0_array_temp = np.linspace(R_0_array[f_ind[i]], R_0_array[f_ind[i] + 1], nR)
        for i_r in range(nR):
            f_kR_re[i_r] = pss.f_kR(R_0_array_temp[i_r], k_array[ik], params)
            # g_kR_re[i] = pss.g_kR(R_0_array[i], k_array[ik], params)

        f_ind_temp = np.where(f_kR_re[:-1] * f_kR_re[1:]<0)[0]
        # g_ind = np.where(g_kR_re[:-1] * g_kR_re[1:]<0)[0]
        # Rf_root[ik, i] = (f_kR_re[f_ind[i] + 1] * R_0_array[f_ind[i]] - f_kR_re[f_ind[i]] * R_0_array[f_ind[i] + 1]) / (f_kR_re[f_ind[i] + 1] - f_kR_re[f_ind[i]])
        Rf_temp[i] = (f_kR_re[f_ind_temp[0] + 1] * R_0_array_temp[f_ind_temp[0]] - f_kR_re[f_ind_temp[0]] * R_0_array_temp[f_ind_temp[0] + 1]) / (f_kR_re[f_ind_temp[0] + 1] - f_kR_re[f_ind_temp[0]])
        
        
    for i in range(len(g_ind)):

        R_0_array_temp = np.linspace(R_0_array[g_ind[i]], R_0_array[g_ind[i] + 1], nR)
        for i_r in range(nR):
            g_kR_re[i_r] = pss.g_kR(R_0_array_temp[i_r], k_array[ik], params)
        g_ind_temp = np.where(g_kR_re[:-1] * g_kR_re[1:]<0)[0]
        Rg_temp[i] = (g_kR_re[g_ind_temp[0] + 1] * R_0_array_temp[g_ind_temp[0]] - g_kR_re[g_ind_temp[0]] * R_0_array_temp[g_ind_temp[0] + 1]) / (g_kR_re[g_ind_temp[0] + 1] - g_kR_re[g_ind_temp[0]])
        
    Rf_root[ik, :] = Rf_temp[:]
    Rg_root[ik, :] = Rg_temp[:]
plt.axhline(0, c='k')
plt.show()

# 用 clac_Inclination 的 nan 位置，遮罩掉 chi_square 對應值
valid_mask = ~np.isnan(clac_Inclination).ravel()

for i_root in range(4):
    # ====== 新增：找出 chi_square 前50小的值和位置（NaN 已被遮掉） ======
    chi_flat = chi_square.ravel()
    sorted_indices_1d = np.argsort(chi_flat[valid_mask])[:n_min]
    valid_indices_1d = np.where(valid_mask)[0][sorted_indices_1d]
    sorted_indices_2d = np.array(np.unravel_index(valid_indices_1d, chi_square.shape)).T

    # 儲存
    chi_square_min[i_root] = sorted_indices_2d

    # 畫紅色 x
for i, j in sorted_indices_2d:
    plt.scatter(k_array[i], j * np.pi / chi_square.shape[1], marker='x', color='r', s=30, linewidths=2)

# 畫 chi_square 背景圖
chi_square_im = plt.imshow(chi_square.T, origin='lower',
                            extent=[k_array[0], k_array[-1], 0, np.pi],
                            aspect=k_array[-1] / np.pi)
plt.ylim(0, np.pi)
plt.xlabel("k")
plt.ylabel("kR")
plt.colorbar(chi_square_im, label='Chi-square')
plt.show()

    # 畫紅色 x
for i, j in sorted_indices_2d:
    plt.scatter(k_array[i], j * np.pi / chi_square.shape[1], marker='x', color='r', s=30, linewidths=2)
inclina_im = plt.imshow(clac_Inclination.T, origin='lower', 
                        extent=[k_array[0], k_array[-1], 0, np.pi],
                        aspect=k_array[-1] / np.pi)

plt.ylim(0, np.pi)
plt.xlabel("k")
plt.ylabel("kR")
plt.colorbar(inclina_im, label='Inclination')
plt.show()

    # 畫紅色 x
for i, j in sorted_indices_2d:
    plt.scatter(k_array[i], j * np.pi / chi_square.shape[1], marker='x', color='r', s=30, linewidths=2)
phi_im = plt.imshow(clac_phi_par.T, origin='lower', 
                        extent=[k_array[0], k_array[-1], 0, np.pi],
                        aspect=k_array[-1] / np.pi)

plt.ylim(0, np.pi)
plt.xlabel("k")
plt.ylabel("kR")
plt.colorbar(phi_im, label='phi')
plt.show()

    # 畫紅色 x
for i, j in sorted_indices_2d:
    plt.scatter(k_array[i], j * np.pi / chi_square.shape[1], marker='x', color='r', s=30, linewidths=2)
theta_im = plt.imshow(clac_theta_par.T, origin='lower', 
                        extent=[k_array[0], k_array[-1], 0, np.pi],
                        aspect=k_array[-1] / np.pi)

plt.ylim(0, np.pi)
plt.xlabel("k")
plt.ylabel("kR")
plt.colorbar(theta_im, label='theta')
plt.show()

    # 畫紅色 x
for i, j in sorted_indices_2d:
    plt.scatter(k_array[i], j * np.pi / chi_square.shape[1], marker='x', color='r', s=30, linewidths=2)
omega_im = plt.imshow(clac_omega_ref.T, origin='lower', 
                        extent=[k_array[0], k_array[-1], 0, np.pi],
                        aspect=k_array[-1] / np.pi)

plt.ylim(0, np.pi)
plt.xlabel("k")
plt.ylabel("kR")
plt.colorbar(omega_im, label='omega')
plt.show()

    # 畫紅色 x
for i, j in sorted_indices_2d:
    plt.scatter(k_array[i], j * np.pi / chi_square.shape[1], marker='x', color='r', s=30, linewidths=2)
t_im = plt.imshow(clac_t.T, origin='lower', 
                        extent=[k_array[0], k_array[-1], 0, np.pi],
                        aspect=k_array[-1] / np.pi)

plt.ylim(0, np.pi)
plt.xlabel("k")
plt.ylabel("kR")
plt.colorbar(t_im, label='t')
plt.show()

for i in range(n_min):
    R_0_array = np.linspace(0, np.pi / k_array[int(chi_square_min[0,i,0])], nR)
    A, B, C, inclin, phi_par, theta_par, omega_ref, t = pss.calc_params(R_0_array[int(chi_square_min[0,i,1])], k_array[int(chi_square_min[0,i,0])], params, radius_ref_au)
    print(i, 'A B C R k')
    print(A, B, C, R_0_array[int(chi_square_min[0,i,1])], k_array[int(chi_square_min[0,i,0])], chi_square[int(chi_square_min[0,i,0]), int(chi_square_min[0,i,1])])
    print('Parameters')
    print(np.rad2deg(inclin), np.rad2deg(theta_par), omega_ref, t / 86400 / 365 / 1e6, np.rad2deg(phi_par))
    print(np.rad2deg(clac_Inclination[int(chi_square_min[0,i,0]), int(chi_square_min[0,i,1])]), np.rad2deg(clac_theta_par[int(chi_square_min[0,i,0]), int(chi_square_min[0,i,1])]), clac_omega_ref[int(chi_square_min[0,i,0]), int(chi_square_min[0,i,1])], clac_t[int(chi_square_min[0,i,0]), int(chi_square_min[0,i,1])] / 86400 / 365 / 1e6, np.rad2deg(clac_phi_par[int(chi_square_min[0,i,0]), int(chi_square_min[0,i,1])]))

    # plt.plot(R[:-6], X_r[:-6])
    # plt.plot(R[:-6], Z_r[:-6])
    # plt.plot(R[:-6], A * np.cos(k_array[int(chi_square_min[0,i,0])] * ( R[:-6] + R_0_array[int(chi_square_min[0,i,1])])))
    # plt.plot(R[:-6], B * np.sin(k_array[int(chi_square_min[0,i,0])] * ( R[:-6] + R_0_array[int(chi_square_min[0,i,1])])) + C)
    # plt.show()
    # **使用最佳參數計算最終擬合結果**
    Theta_best = theta_par
    Phi_best = phi_par
    T_best = t / 86400 / 365 / 1e6
    Inclination_best = inclin
    Omega_best = omega_ref
    x_best, y_best, z_best, u_best, v_best, w_best = pss.PSS_model(Theta_best, Phi_best, Inclination_best, radius_ref_au, 
                                                                T_best, Omega_best, 2.5e3, 7.5e3, 50)

    # **確保 `PSS_model` 的輸出也是 AU 單位**
    # x_best, y_best, v_best = np.array(x_best), np.array(y_best), np.array(v_rotat)  # AU 單位

    x_fitresult = x_best * np.cos(pa_rad) - z_best * np.sin(pa_rad)
    z_fitresult = x_best * np.sin(pa_rad) + z_best * np.cos(pa_rad)

    x_means_ori_AU = x_means * AU_per_pixel  # 以影像中心 (128, 128) 為基準
    z_means_ori_AU = z_means * AU_per_pixel  # 以影像中心 (128, 128) 為基準


    x_axis_range = [-10000, 10000]
    y_axis_range = [-10000, 10000]
    z_axis_range = [-1, 1]

    # **畫出 3D 圖**
    fig = go.Figure()

    # **Obs-line 觀測點（目標曲線）**
    fig.add_trace(go.Scatter3d(
        x=x_means_ori_AU, y=z_means_ori_AU, z=v_means_LS_km,
        mode='markers',
        marker=dict(
            size=marker_size, 
            color=v_means_LS_km, 
            colorscale='RdBu_r',
            cauto=False, cmin=-0.5, cmax=0.5,
            opacity=0.5,
        ),
        name="Obs-Strline"
    ))

    # **PSS Model 擬合曲線（最佳匹配）**
    # fig.add_trace(go.Scatter3d(
    #     x=x_fitresult, y=y_fitresult, z=v_best,
    #     mode='markers',
    #     marker=dict(
    #         size=marker_size, 
    #         color=v_best, 
    #         colorscale='RdBu_r',
    #         cauto=False, cmin=-0.5, cmax=0.6,
    #         opacity=0.5,
    #     ),
    #     name="Best Fit Model"
    # ))
    fig.add_trace(go.Scatter3d(
        x=x_fitresult, y=z_fitresult, z=v_best,
        mode='lines',
        line=dict(color='blue', width=4),
        name="Best Fit Model"
    ))

    # **添加原始觀測數據**
    # fig.add_trace(go.Scatter3d(
    #     x=x_AU, y=y_AU, z=z_LS,
    #     mode='markers',
    #     marker=dict(
    #         size=marker_size, 
    #         color=z_LS, 
    #         colorscale='RdBu_r',
    #         cauto=False, cmin=-0.5, cmax=0.6,
    #         opacity=0.5,
    #         colorbar=dict(title="(km/s)", orientation='h', x=0.5, y=-0.4)
    #     ),
    #     name="Original Data"
    # ))

    # **設定 3D 圖軸**
    # fig.update_layout(
    #     title="Obs-Strline Fit vs. PSS Model Best Fit",
    #     scene=dict(
    #         xaxis_title="X (AU)",
    #         yaxis_title="Y (AU)",
    #         zaxis_title="Z (km/s)",
    #         xaxis = dict(nticks=4, range=x_axis_range,),
    #         yaxis = dict(nticks=4, range=y_axis_range,),
    #         zaxis = dict(nticks=4, range=z_axis_range,),
    #         aspectmode="manual",
    #         aspectratio=dict(x=1, y=1, z=1)
    #     ),
    #     margin=dict(l=0, r=0, b=0, t=40),
    # )
    # fig.update_traces(colorbar = dict(orientation='h', y = -0.25, x = 0.5))

    fig.show()




# for i_root in range(4):
#     # 畫根的位置
#     plt.scatter(k_array, Rf_root[:, i_root] * k_array, c='C0', s=10)
#     plt.scatter(k_array, Rg_root[:, i_root] * k_array, c='C1', s=10)


# plt.imshow(clac_Inclination.T, origin='lower',)
# plt.show()
# for i_root in range(4):
#     # 畫根的位置
#     plt.scatter(k_array, Rf_root[:, i_root] * k_array, c='C0', s=10)
#     plt.scatter(k_array, Rg_root[:, i_root] * k_array, c='C1', s=10)
# fig = plt.figure(figsize=(8.27, 8.27))
# ax  = fig.add_subplot(111)